# Test seasons

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rubin_sim.maf as maf

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

## Configuration

In [ ]:
# Baseline Survey
opsdb_fname = get_baseline()
run_name = os.path.split(opsdb_fname)[-1].replace(".db", "")

print(f"Using {run_name}, to be read from {opsdb_fname}")

print(run_name)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="03_testseasons_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

## Slicer, Metrics and Bundle

In [ ]:
# Set up output
out_dir = data_dir

nside = 16
slicer = maf.HealpixSlicer(nside=nside, use_cache=True)
metric = maf.SeasonLengthMetric()
m2 = maf.CountMetric(col="night")
bundle = maf.MetricBundle(metric, slicer, None, run_name=run_name)
b2 = maf.MetricBundle(m2, slicer, None)

bg = maf.MetricBundleGroup({"season": bundle, "night": b2}, opsdb_fname, out_dir)


# Behind the scenes stuff to get the simulated data and set up the slicer so we can test ONE point
bg.set_current("")
bg.get_data("")
simData = bg.sim_data
bundle.slicer.setup_slicer(simData)

In [ ]:
# Find a good spot on the sky with lots of visits
lenvisits = []
for s in bundle.slicer:
    lenvisits.append(len(s["idxs"]))
lenvisits = np.array(lenvisits)
x = np.where(lenvisits == np.max(lenvisits))[0][0]

bundle.slicer[x]

In [ ]:
from astropy.time import Time
from astropy.coordinates import get_sun
from astropy.coordinates import EarthLocation

loc = EarthLocation.of_site("Cerro Pachon")
t = Time("2026-09-22T00:00:00.00", format="isot", scale="utc", location=loc)
print("Time-mjd : ", t.utc.mjd)
print("sun RA and mjd ", get_sun(t).ra.deg, t.utc.mjd)
print("local sidereal time to adjust to season start", t.sidereal_time("mean").deg)

In [ ]:
def calcSeason(ra, time):
    """Calculate the 'season' in the survey for a series of ra/time values of an observation.
    Based only on the RA of the point on the sky, it calculates the 'season' based on when the sun
    passes through this RA (this marks the start of a 'season').

    Note that seasons should be calculated using the RA of a fixed point on the sky, such as
    the slicePoint['ra'] if calculating season values for a series of opsim pointings on the sky.
    To convert to integer seasons, use np.floor(seasons)

    Parameters
    ----------
    ra : `float`
        The RA (in degrees) of the point on the sky
    time : `np.ndarray`
        The times of the observations, in MJD days

    Returns
    -------
    seasons : `np.array`
        The season values, as floats.
    """
    # A reference time and sun RA location to anchor the location of the Sun
    # This time was chosen as it is close to the expected start of the survey.
    refTime = 61305.0
    refSunRA = 178.7547619850774
    # Calculate the fraction of the sphere/"year" for this location
    offset = (ra - refSunRA) / 360 * 365.25
    # Calculate when the seasons should begin
    seasonBegan = refTime + offset
    # Calculate the season value for each point.
    seasons = (time - seasonBegan) / 365.25
    # (usually) Set first season at this point to 0
    seasons = seasons - np.floor(np.min(seasons))
    return seasons
    # The reference values can be evaluated using:
    # from astropy.time import Time
    # from astropy.coordinates import get_sun
    # from astropy.coordinates import EarthLocation
    # loc = EarthLocation.of_site('Cerro Pachon')
    # t = Time('2024-09-22T00:00:00.00', format='isot', scale='utc', location=loc)
    # print('Ref time', t.utc.mjd)
    # print('Ref sun RA', get_sun(t).ra.deg, t.utc.mjd)
    # print('local sidereal time at season start', t.sidereal_time('apparent').deg)

In [ ]:
# x = 2593
# x = 2999
# x = 890
x = 2645
obsidx = slicer[x]["idxs"]
ss = np.sort(simData[obsidx], order="observationStartMJD")
seasons = calcSeason(np.degrees(slicer[x]["slice_point"]["ra"]), ss["observationStartMJD"])
intseason = np.floor(seasons)
print(
    len(ss),
    ss["fieldRA"].min(),
    ss["fieldRA"].max(),
    np.degrees(slicer[x]["slice_point"]["ra"]),
    ss["fieldDec"].mean(),
    len(obsidx),
)

In [ ]:
plt.figure(figsize=(8, 5))
for s in np.unique(intseason):
    match = np.where(intseason == s)
    plt.plot(seasons[match], ss["night"][match], linestyle="", marker=".")
for n in range(0, 11):
    plt.axvline(n, alpha=0.3)

In [ ]:
seasongaps = 106
xx = np.where(np.diff(ss["night"]) > seasongaps)[0]
qq = np.where(np.diff(np.floor(seasons)) > 0)[0]
xx, qq, np.floor(seasons)
pd.DataFrame(
    [
        np.diff(ss["night"])[xx],
        ss["night"][xx - 1],
        ss["night"][xx],
        ss["night"][xx + 1],
        seasons[xx - 1],
        seasons[xx],
        seasons[xx + 1],
    ],
    index=["delta night", "n-1", "n", "n+1", "season-1", "season", "season+1"],
)

## Run Metrics

In [ ]:
bg.run_all()

## Plot Metrics

In [ ]:
bundle.plot()

# Check every point against gaps in nights

In [ ]:
# Check every point against gaps in nights
for s in slicer:
    obsidx = s["idxs"]
    if len(obsidx) > 3:
        ss = np.sort(simData[obsidx], order="observationStartMJD")
        seasons = calcSeason(np.degrees(s["slice_point"]["ra"]), ss["observationStartMJD"])
        intseasons = np.floor(seasons)
        xx = np.where(np.diff(ss["night"]) > seasongaps)[0]
        qq = np.where(np.diff(intseasons) > 0)[0]

        if not np.all(xx == qq):
            print(s["slice_point"]["sid"])